# Notebook 06b — Robustness check: sensibilità a k (k=12, k=18)

**Obiettivo:** verificare se il profilo "hub anomalo ad alta betweenness" (C0/C10 in k=15)
è stabile al variare di k, o è un artefatto della scelta k=15.

**Pipeline identica a 06:** stessa feature matrix, stesso scaler, stesso set 15 feature comportamentali.
Nessuna modifica al notebook originale.

**Criteri classificazione hub-anomalo:**
- betweenness z-score > 1.5 (sopra la media)
- egonet_density z-score < 0 (tendenzialmente bassa)

**Output:** tabella di confronto {k, n_cluster_hub, numerosità, betweenness_z, burstiness_z, ARI_vs_k15}

## 0. Setup

In [1]:
import warnings
import numpy as np
import pandas as pd
from scipy.stats import zscore
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

warnings.filterwarnings('ignore')
np.random.seed(42)

VOLUME_FEATURES = ["n_posts", "n_comments", "n_comments_received", "active_days"]

# Soglie per identificare cluster "hub anomalo" (stesso profilo C0/C10)
BETWEENNESS_Z_THRESHOLD = 1.5   # betweenness z-score >= threshold
EGONET_DENSITY_Z_MAX    = 0.0   # egonet_density z-score < threshold (struttura rada)

print("Setup OK")

Setup OK


## 1. Caricamento dati

In [2]:
df_scaled = pd.read_parquet("../data/feature_matrix_scaled_v1.parquet")
df_raw    = pd.read_parquet("../data/feature_matrix_graph_v1.parquet")

agent_ids      = df_scaled["agent_id"].values
all_features   = [c for c in df_scaled.columns if c != "agent_id"]
behav_features = [f for f in all_features if f not in VOLUME_FEATURES]

X = df_scaled[behav_features].values   # (9096, 15)
N = len(X)

assert np.isnan(X).sum() == 0, "NaN rilevati!"
print(f"Feature comportamentali ({len(behav_features)}): {behav_features}")
print(f"X shape: {X.shape} — NaN: 0")

Feature comportamentali (15): ['burstiness_posts', 'hour_entropy', 'reply_to_post_ratio', 'self_reply_rate', 'mean_thread_depth', 'mean_post_length', 'std_post_length', 'type_token_ratio', 'in_degree', 'out_degree', 'pagerank', 'betweenness', 'local_clustering', 'egonet_density', 'reciprocity_local']
X shape: (9096, 15) — NaN: 0


## 2. Caricamento partizione k=15 originale (riferimento)

In [3]:
df_k15 = pd.read_parquet("../data/cluster_assignments_v1.parquet")
# Allinea all'ordine di agent_ids in X
id_to_label_k15 = dict(zip(df_k15["agent_id"], df_k15["cluster_id"]))
labels_k15      = np.array([id_to_label_k15[aid] for aid in agent_ids])

print(f"Partizione k=15 caricata: {len(labels_k15)} agenti")
unique, counts = np.unique(labels_k15, return_counts=True)
print(f"Cluster k=15 — min={counts.min()}, max={counts.max()}, median={np.median(counts):.0f}")

Partizione k=15 caricata: 9096 agenti
Cluster k=15 — min=172, max=2346, median=421


## 3. Funzione: profilo cluster + identificazione hub-anomali

In [4]:
def run_kmeans_and_profile(k, X, agent_ids, df_raw, behav_features, all_features,
                           seeds=(42, 123, 999)):
    """
    Esegue k-means con 3 seed, sceglie il seed con silhouette massima.
    Restituisce labels finali, silhouette, seed scelto, e DataFrame profili z-score.
    """
    np.random.seed(42)
    sil_idx = np.random.choice(len(X), min(2000, len(X)), replace=False)
    
    best_sil, best_labels, best_seed = -1, None, None
    for seed in seeds:
        km  = KMeans(n_clusters=k, random_state=seed, n_init=10)
        lbl = km.fit_predict(X)
        sil = silhouette_score(X[sil_idx], lbl[sil_idx])
        if sil > best_sil:
            best_sil, best_labels, best_seed = sil, lbl, seed
    
    # Profili sulle feature raw (non scalate) in z-score cross-cluster
    df_merged = df_raw[["agent_id"] + all_features].copy()
    df_merged["cluster_id"] = [dict(zip(agent_ids, best_labels))[aid] for aid in df_merged["agent_id"]]
    
    cluster_means = df_merged.groupby("cluster_id")[all_features].mean()
    # z-score dei profili medi (tra cluster): ogni colonna normalizzata
    profiles_z = cluster_means.apply(zscore, axis=0)
    
    cluster_sizes = pd.Series(best_labels).value_counts().rename("size")
    
    return best_labels, best_sil, best_seed, profiles_z, cluster_means, cluster_sizes


def identify_hub_anomalous(profiles_z, cluster_sizes,
                           betweenness_z_threshold=1.5,
                           egonet_density_z_max=0.0):
    """
    Identifica cluster con profilo 'hub anomalo':
    - betweenness z-score >= betweenness_z_threshold
    - egonet_density z-score < egonet_density_z_max
    """
    hub_clusters = []
    for cid in profiles_z.index:
        bw_z  = profiles_z.loc[cid, "betweenness"]
        eg_z  = profiles_z.loc[cid, "egonet_density"]
        if bw_z >= betweenness_z_threshold and eg_z < egonet_density_z_max:
            hub_clusters.append({
                "cluster_id":      f"C{cid}",
                "n":               cluster_sizes.get(cid, 0),
                "betweenness_z":   round(float(bw_z), 3),
                "burstiness_z":    round(float(profiles_z.loc[cid, "burstiness_posts"]), 3),
                "egonet_density_z": round(float(eg_z), 3),
                "out_degree_z":    round(float(profiles_z.loc[cid, "out_degree"]), 3),
                "pagerank_z":      round(float(profiles_z.loc[cid, "pagerank"]), 3),
            })
    return pd.DataFrame(hub_clusters).sort_values("betweenness_z", ascending=False)


print("Funzioni definite.")

Funzioni definite.


## 4. Run k=12

In [5]:
print("=" * 60)
print("k = 12")
print("=" * 60)

labels_k12, sil_k12, seed_k12, profiles_z_k12, cluster_means_k12, sizes_k12 = run_kmeans_and_profile(
    k=12, X=X, agent_ids=agent_ids, df_raw=df_raw,
    behav_features=behav_features, all_features=all_features
)

ari_k12 = adjusted_rand_score(labels_k15, labels_k12)

print(f"Silhouette: {sil_k12:.4f} | Seed: {seed_k12}")
print(f"ARI vs k=15: {ari_k12:.4f}")
print(f"\nDimensioni cluster k=12:")
for cid, sz in sizes_k12.sort_values(ascending=False).items():
    print(f"  C{cid}: {sz} ({sz/N*100:.1f}%)")

k = 12


Silhouette: 0.2599 | Seed: 999
ARI vs k=15: 0.9107

Dimensioni cluster k=12:
  C9: 2345 (25.8%)
  C5: 1551 (17.1%)
  C11: 776 (8.5%)
  C0: 732 (8.0%)
  C2: 660 (7.3%)
  C7: 632 (6.9%)
  C1: 516 (5.7%)
  C10: 484 (5.3%)
  C6: 437 (4.8%)
  C3: 405 (4.5%)
  C4: 388 (4.3%)
  C8: 170 (1.9%)


In [6]:
hub_k12 = identify_hub_anomalous(profiles_z_k12, sizes_k12,
                                  BETWEENNESS_Z_THRESHOLD, EGONET_DENSITY_Z_MAX)

print(f"Cluster hub-anomali in k=12 (betweenness_z >= {BETWEENNESS_Z_THRESHOLD}, egonet_density_z < {EGONET_DENSITY_Z_MAX}):")
print(f"Trovati: {len(hub_k12)}")
print(hub_k12.to_string(index=False))

Cluster hub-anomali in k=12 (betweenness_z >= 1.5, egonet_density_z < 0.0):
Trovati: 1
cluster_id   n  betweenness_z  burstiness_z  egonet_density_z  out_degree_z  pagerank_z
        C1 516          1.998         1.357            -0.421         2.236       1.999


In [7]:
# Profili completi betweenness (tutti i cluster, ordinati)
top_bw_k12 = profiles_z_k12[["betweenness", "burstiness_posts", "egonet_density",
                               "out_degree", "pagerank", "reciprocity_local"]].copy()
top_bw_k12.index = [f"C{i}" for i in top_bw_k12.index]
top_bw_k12["n"] = [sizes_k12.get(int(i[1:]), 0) for i in top_bw_k12.index]
top_bw_k12 = top_bw_k12.sort_values("betweenness", ascending=False)
print("\nTutti i cluster k=12 — z-score key features (ordinati per betweenness):")
print(top_bw_k12.round(3).to_string())


Tutti i cluster k=12 — z-score key features (ordinati per betweenness):
     betweenness  burstiness_posts  egonet_density  out_degree  pagerank  reciprocity_local     n
C1         1.998             1.357          -0.421       2.236     1.999              0.272   516
C10        1.752            -0.126           0.654       1.305     1.792              1.680   484
C2         1.326             0.088          -0.152       1.035     0.106             -0.148   660
C8        -0.110            -2.402           0.006      -0.069    -0.336             -0.145   170
C7        -0.473            -0.040           2.837      -0.240    -0.054              2.510   632
C3        -0.517             1.655          -0.349      -0.481     0.038             -0.406   405
C4        -0.522            -1.030          -0.344      -0.342    -0.703             -0.518   388
C11       -0.577             0.741          -0.355      -0.226    -0.737             -0.577   776
C5        -0.694            -0.076           

## 5. Run k=18

In [8]:
print("=" * 60)
print("k = 18")
print("=" * 60)

labels_k18, sil_k18, seed_k18, profiles_z_k18, cluster_means_k18, sizes_k18 = run_kmeans_and_profile(
    k=18, X=X, agent_ids=agent_ids, df_raw=df_raw,
    behav_features=behav_features, all_features=all_features
)

ari_k18 = adjusted_rand_score(labels_k15, labels_k18)

print(f"Silhouette: {sil_k18:.4f} | Seed: {seed_k18}")
print(f"ARI vs k=15: {ari_k18:.4f}")
print(f"\nDimensioni cluster k=18:")
for cid, sz in sizes_k18.sort_values(ascending=False).items():
    print(f"  C{cid}: {sz} ({sz/N*100:.1f}%)")

k = 18


Silhouette: 0.2625 | Seed: 999
ARI vs k=15: 0.8614

Dimensioni cluster k=18:
  C4: 2252 (24.8%)
  C1: 1405 (15.4%)
  C14: 698 (7.7%)
  C11: 557 (6.1%)
  C13: 492 (5.4%)
  C6: 443 (4.9%)
  C3: 417 (4.6%)
  C2: 385 (4.2%)
  C7: 381 (4.2%)
  C10: 368 (4.0%)
  C8: 340 (3.7%)
  C17: 334 (3.7%)
  C12: 243 (2.7%)
  C16: 204 (2.2%)
  C0: 189 (2.1%)
  C9: 187 (2.1%)
  C15: 104 (1.1%)
  C5: 97 (1.1%)


In [9]:
hub_k18 = identify_hub_anomalous(profiles_z_k18, sizes_k18,
                                  BETWEENNESS_Z_THRESHOLD, EGONET_DENSITY_Z_MAX)

print(f"Cluster hub-anomali in k=18 (betweenness_z >= {BETWEENNESS_Z_THRESHOLD}, egonet_density_z < {EGONET_DENSITY_Z_MAX}):")
print(f"Trovati: {len(hub_k18)}")
print(hub_k18.to_string(index=False))

Cluster hub-anomali in k=18 (betweenness_z >= 1.5, egonet_density_z < 0.0):
Trovati: 2
cluster_id   n  betweenness_z  burstiness_z  egonet_density_z  out_degree_z  pagerank_z
        C0 189          2.043         0.175            -0.619         2.162       2.584
       C12 243          1.874         1.559            -0.544         2.098       2.214


In [10]:
top_bw_k18 = profiles_z_k18[["betweenness", "burstiness_posts", "egonet_density",
                               "out_degree", "pagerank", "reciprocity_local"]].copy()
top_bw_k18.index = [f"C{i}" for i in top_bw_k18.index]
top_bw_k18["n"] = [sizes_k18.get(int(i[1:]), 0) for i in top_bw_k18.index]
top_bw_k18 = top_bw_k18.sort_values("betweenness", ascending=False)
print("\nTutti i cluster k=18 — z-score key features (ordinati per betweenness):")
print(top_bw_k18.round(3).to_string())


Tutti i cluster k=18 — z-score key features (ordinati per betweenness):
     betweenness  burstiness_posts  egonet_density  out_degree  pagerank  reciprocity_local     n
C0         2.043             0.175          -0.619       2.162     2.584              0.454   189
C12        1.874             1.559          -0.544       2.098     2.214              0.202   243
C15        1.155            -1.177          -0.114       1.058     0.457              0.204   104
C8         1.042            -0.020           1.274       0.346     0.758              2.066   340
C13        1.023            -0.037          -0.175       0.580    -0.131             -0.333   492
C2         1.017             0.876          -0.191       1.059     0.251              0.141   385
C17       -0.351            -0.036           1.399       0.124    -0.366              1.335   334
C5        -0.398            -2.532           0.057      -0.421    -0.467             -0.236    97
C16       -0.584             1.800          -

## 6. Referimento: profilo C0/C10 in k=15

In [11]:
# Ricostruisci profili k=15 per confronto
df_merged_k15 = df_raw[["agent_id"] + all_features].copy()
df_merged_k15["cluster_id"] = [id_to_label_k15[aid] for aid in df_merged_k15["agent_id"]]
cluster_means_k15 = df_merged_k15.groupby("cluster_id")[all_features].mean()
profiles_z_k15    = cluster_means_k15.apply(zscore, axis=0)
sizes_k15         = pd.Series(labels_k15).value_counts()

hub_k15 = identify_hub_anomalous(profiles_z_k15, sizes_k15,
                                  BETWEENNESS_Z_THRESHOLD, EGONET_DENSITY_Z_MAX)

print("Cluster hub-anomali in k=15 (riferimento):")
print(hub_k15.to_string(index=False))

Cluster hub-anomali in k=15 (riferimento):
cluster_id   n  betweenness_z  burstiness_z  egonet_density_z  out_degree_z  pagerank_z
        C0 213          2.012         0.022            -0.599         2.105       2.405
       C10 253          1.840         1.687            -0.530         2.011       2.033


## 7. Tabella di confronto finale (output per tesi/slide)

In [12]:
def build_summary_row(k, hub_df, ari_vs_k15, sil):
    """Riga riassuntiva per la tabella di confronto."""
    if len(hub_df) == 0:
        return {
            "k": k,
            "silhouette": round(sil, 4),
            "ARI_vs_k15": round(ari_vs_k15, 4) if ari_vs_k15 is not None else "—",
            "n_cluster_hub": 0,
            "cluster_ids": "—",
            "numerosità_totale": 0,
            "betweenness_z_mean": "—",
            "burstiness_z_mean": "—",
            "egonet_density_z_mean": "—",
        }
    return {
        "k": k,
        "silhouette": round(sil, 4),
        "ARI_vs_k15": round(ari_vs_k15, 4) if ari_vs_k15 is not None else "—",
        "n_cluster_hub": len(hub_df),
        "cluster_ids": ", ".join(hub_df["cluster_id"].tolist()),
        "numerosità_totale": int(hub_df["n"].sum()),
        "betweenness_z_mean": round(hub_df["betweenness_z"].mean(), 3),
        "burstiness_z_mean":  round(hub_df["burstiness_z"].mean(), 3),
        "egonet_density_z_mean": round(hub_df["egonet_density_z"].mean(), 3),
    }

rows = [
    build_summary_row(15, hub_k15, None,   sil=0.2659),  # k=15: ARI riferimento
    build_summary_row(12, hub_k12, ari_k12, sil_k12),
    build_summary_row(18, hub_k18, ari_k18, sil_k18),
]

df_comparison = pd.DataFrame(rows).set_index("k")

print("=" * 80)
print("TABELLA DI ROBUSTEZZA — Cluster hub-anomali (betweenness alta, egonet_density bassa)")
print(f"Criteri: betweenness_z >= {BETWEENNESS_Z_THRESHOLD} AND egonet_density_z < {EGONET_DENSITY_Z_MAX}")
print("=" * 80)
print(df_comparison.to_string())
print("="*80)

TABELLA DI ROBUSTEZZA — Cluster hub-anomali (betweenness alta, egonet_density bassa)
Criteri: betweenness_z >= 1.5 AND egonet_density_z < 0.0
    silhouette ARI_vs_k15  n_cluster_hub cluster_ids  numerosità_totale  betweenness_z_mean  burstiness_z_mean  egonet_density_z_mean
k                                                                                                                                    
15      0.2659          —              2     C0, C10                466               1.926              0.854                 -0.564
12      0.2599     0.9107              1          C1                516               1.998              1.357                 -0.421
18      0.2625     0.8614              2     C0, C12                432               1.959              0.867                 -0.582


In [13]:
# Dettaglio cluster hub per ogni k
for k_label, hub_df in [(15, hub_k15), (12, hub_k12), (18, hub_k18)]:
    print(f"\n--- k={k_label}: cluster hub-anomali (dettaglio) ---")
    if len(hub_df) == 0:
        print("  Nessun cluster soddisfa i criteri.")
    else:
        print(hub_df.to_string(index=False))


--- k=15: cluster hub-anomali (dettaglio) ---
cluster_id   n  betweenness_z  burstiness_z  egonet_density_z  out_degree_z  pagerank_z
        C0 213          2.012         0.022            -0.599         2.105       2.405
       C10 253          1.840         1.687            -0.530         2.011       2.033

--- k=12: cluster hub-anomali (dettaglio) ---
cluster_id   n  betweenness_z  burstiness_z  egonet_density_z  out_degree_z  pagerank_z
        C1 516          1.998         1.357            -0.421         2.236       1.999

--- k=18: cluster hub-anomali (dettaglio) ---
cluster_id   n  betweenness_z  burstiness_z  egonet_density_z  out_degree_z  pagerank_z
        C0 189          2.043         0.175            -0.619         2.162       2.584
       C12 243          1.874         1.559            -0.544         2.098       2.214


## 8. Interpretazione

In [14]:
print("INTERPRETAZIONE ROBUSTEZZA")
print("=" * 60)
print()
print(f"k=15 (riferimento): {len(hub_k15)} cluster hub-anomali")
print(f"  → C0 (n={hub_k15.iloc[0]['n'] if len(hub_k15)>0 else 'N/A'}): betweenness_z={'N/A' if len(hub_k15)==0 else hub_k15.iloc[0]['betweenness_z']}, burstiness_z={'N/A' if len(hub_k15)==0 else hub_k15.iloc[0]['burstiness_z']}")
if len(hub_k15) > 1:
    print(f"  → C10 (n={hub_k15.iloc[1]['n']}): betweenness_z={hub_k15.iloc[1]['betweenness_z']}, burstiness_z={hub_k15.iloc[1]['burstiness_z']}")
print()
print(f"k=12: {len(hub_k12)} cluster hub-anomali | ARI vs k=15 = {ari_k12:.4f}")
print(f"k=18: {len(hub_k18)} cluster hub-anomali | ARI vs k=15 = {ari_k18:.4f}")
print()

if len(hub_k12) > 0 and len(hub_k18) > 0:
    print("CONCLUSIONE: il profilo hub-anomalo è STABILE al variare di k.")
    print("I cluster con betweenness alta + egonet_density bassa emergono in tutti e tre i run.")
    print("Questo supporta l'ipotesi che C0/C10 non siano artefatti di k=15.")
elif len(hub_k12) == 0 or len(hub_k18) == 0:
    print("ATTENZIONE: il profilo hub-anomalo NON emerge in tutti i run.")
    print("Verificare le soglie o se il profilo si frammenta/fonde a k diversi.")
print()
print("Nota: ARI alto (>0.7) indica che le partizioni sono sostanzialmente compatibili.")
print("ARI vs k=15 per k vicini (~0.6–0.9) è atteso: cambio k frammenta/fonde cluster confinanti.")

INTERPRETAZIONE ROBUSTEZZA

k=15 (riferimento): 2 cluster hub-anomali
  → C0 (n=213): betweenness_z=2.012, burstiness_z=0.022
  → C10 (n=253): betweenness_z=1.84, burstiness_z=1.687

k=12: 1 cluster hub-anomali | ARI vs k=15 = 0.9107
k=18: 2 cluster hub-anomali | ARI vs k=15 = 0.8614

CONCLUSIONE: il profilo hub-anomalo è STABILE al variare di k.
I cluster con betweenness alta + egonet_density bassa emergono in tutti e tre i run.
Questo supporta l'ipotesi che C0/C10 non siano artefatti di k=15.

Nota: ARI alto (>0.7) indica che le partizioni sono sostanzialmente compatibili.
ARI vs k=15 per k vicini (~0.6–0.9) è atteso: cambio k frammenta/fonde cluster confinanti.
